# Lab 01: How a Spark application runs

**Time:** about 2 h 15 min.

**Checklist questions this lab gives you evidence for:**<br><span style="color:#d93025"><b>1.1 Apache Spark architecture</b></span><br><span style="color:#d93025"><b>1.2 What is a SparkSession?</b></span><br><span style="color:#d93025"><b>1.3 Driver vs. executor</b></span><br><span style="color:#d93025"><b>1.5 Narrow vs. wide dependencies</b></span><br><span style="color:#d93025"><b>1.7 Fault tolerance</b></span><br><span style="color:#d93025"><b>1.10 Lifecycle of a Spark job</b></span><br><span style="color:#d93025"><b>1.11 Transformation vs. action</b></span><br><span style="color:#d93025"><b>1.12 Lazy evaluation</b></span>

## Data information
NYC Yellow Taxi trips, January 2024 ([source](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page), [data dictionary](https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf)): 2,964,624 trips. The setup cell downloads it. Real data, so some rows are dirty (dates outside January, negative fares).

**trips** (`data/yellow_tripdata_2024-01.parquet`):
* *tpep_pickup_datetime*, *tpep_dropoff_datetime*: when the meter started and stopped
* *PULocationID*, *DOLocationID*: pickup and dropoff zone (joins to *zones.LocationID*)
* *trip_distance*: miles; *passenger_count*: number of passengers
* *payment_type*: 1 = credit card, 2 = cash, 3 = no charge, 4 = dispute, 0 = unknown
* *fare_amount*, *tip_amount*, *total_amount*: dollars
* *VendorID*: the company that sent the record

**zones** (`data/taxi_zone_lookup.csv`, 265 rows): *LocationID*, *Borough*, *Zone*, *service_zone*


## How to work
* Write your answer in the code cell under each question, then run the check cell below it: ✅ correct, ❌ try again.
* Lab 01 gives code skeletons with `___` blanks for the first questions only. From then on, and in every lab after it, you write the code yourself; cells only contain the objects a question gives you.
* When a question says *guess first*, write your guess as a comment before running.
* Keep the Spark UI open (the setup cell prints its address).
* Stuck for more than 10 minutes? Read the linked docs page, then the reference solution.


## Documentation
Everything you need is on these pages. The questions do not repeat them: read, try things out in a cell, and find the answers yourself.

* [SparkSession](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.SparkSession.html)
* [DataFrameReader.parquet](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameReader.parquet.html)
* [DataFrameReader.csv](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameReader.csv.html)
* [unix_timestamp](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.unix_timestamp.html)
* [RDD operations: transformations and actions](https://spark.apache.org/docs/latest/rdd-programming-guide.html#rdd-operations)
* [DataFrame.explain](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.explain.html)
* [DataFrame.agg](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.agg.html)
* [functions.avg](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.avg.html)
* [Spark UI: Stages tab](https://spark.apache.org/docs/latest/web-ui.html#stages-tab)
* [DataFrame.orderBy](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.orderBy.html)
* [DataFrame.groupBy](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.groupBy.html)
* [Tuning partitions](https://spark.apache.org/docs/latest/sql-performance-tuning.html#tuning-partitions)
* [RDD.getNumPartitions](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.RDD.getNumPartitions.html)
* [DataFrame.join](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.join.html)
* [Submitting applications](https://spark.apache.org/docs/latest/submitting-applications.html)
* [functions.udf](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.udf.html)
* [Master URLs](https://spark.apache.org/docs/latest/submitting-applications.html#master-urls)
* [spark.task.maxFailures](https://spark.apache.org/docs/latest/configuration.html#scheduling)

## Setup

In [1]:
# Setup: run once. Downloads the data, starts Spark, defines the helpers the tasks use.
import hashlib, json, os, re, sys, time, urllib.request, contextlib, io
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)   # executors use this kernel's Python
from pyspark.sql import SparkSession, functions as F

os.makedirs("data", exist_ok=True)
for name, url in {
    "yellow_tripdata_2024-01.parquet": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet",
    "taxi_zone_lookup.csv": "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv",
}.items():
    if not os.path.exists(f"data/{name}"):
        print("downloading", name); urllib.request.urlretrieve(url, f"data/{name}")

def new_session(master="local[4]"):
    s = (SparkSession.builder.master(master).appName("day01_lab")
         .config("spark.ui.showConsoleProgress", "false")
         # Spark 4.2.0 bug (SPARK-60294): Python UDFs on Parquet can hang without this.
         .config("spark.sql.execution.arrow.pythonUDF.columnarInput.enabled", "false")
         .getOrCreate())
    s.sparkContext.setLogLevel("ERROR")
    s.conf.set("spark.sql.adaptive.enabled", "false")   # AQE comes in Lab 03; off = exact job/stage counts
    return s

spark = new_session()
print(f"Spark {spark.version} | UI: {spark.sparkContext.uiWebUrl}")

# --- helpers --------------------------------------------------------------------
def _tracker():
    return spark.sparkContext.statusTracker()

def _run_group(action):
    sc = spark.sparkContext
    group = f"g{time.time_ns()}"
    sc.setJobGroup(group, group)
    try:
        action()
    finally:
        sc.setLocalProperty("spark.jobGroup.id", None)
    return [_tracker().getJobInfo(j) for j in sorted(_tracker().getJobIdsForGroup(group))]

def jobs_run(action):
    """How many Spark jobs calling action() starts."""
    return len(_run_group(action))

def stages_run(action):
    """How many stages those jobs had in total."""
    return sum(len(j.stageIds) for j in _run_group(action))

def last_stage_tasks(action):
    """Number of tasks in the last stage action() ran."""
    jobs = _run_group(action)
    last = max(s for j in jobs for s in j.stageIds)
    return _tracker().getStageInfo(last).numTasks

def failed_attempts():
    """Failed task attempts in the most recent failed stage (from the Spark UI's REST API)."""
    sc = spark.sparkContext
    base = f"{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}"
    failed = json.load(urllib.request.urlopen(base + "/stages?status=failed"))
    if not failed:
        return 0
    st = max(failed, key=lambda s: s["stageId"])
    tasks = json.load(urllib.request.urlopen(f"{base}/stages/{st['stageId']}/{st['attemptId']}/taskList?length=500"))
    return sum(1 for t in tasks if t["status"] == "FAILED")

def plan(df, mode="formatted"):
    """explain() output as a string."""
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        df.explain(mode)
    return buf.getvalue()

def optimized(df):
    """The optimized logical plan as a string."""
    return df._jdf.queryExecution().optimizedPlan().toString()

def operators(df):
    """Operator names of the physical plan tree, top to bottom."""
    tree = plan(df).split("\n\n")[0].splitlines()[1:]
    return [re.sub(r"\s*\(\d+\)$", "", line).strip(" +-*:") for line in tree]

def rules_fired(df):
    """Names of the Catalyst rules that changed this query's plan."""
    qe = df._jdf.queryExecution(); qe.executedPlan()
    it, fired = qe.tracker().rules().iterator(), set()
    while it.hasNext():
        kv = it.next()
        if kv._2().numEffectiveInvocations() > 0:
            fired.add(kv._1().split(".")[-1].split("$")[-1])
    return fired

def estimate(df):
    """(sizeInBytes, rowCount or None) the optimizer estimates for df."""
    stats = df._jdf.queryExecution().optimizedPlan().stats()
    rows = int(str(stats.rowCount().get())) if stats.rowCount().isDefined() else None
    return int(str(stats.sizeInBytes())), rows

def timeit(action):
    start = time.perf_counter(); action(); return round(time.perf_counter() - start, 2)

# --- checks: compare a hash of your answer with the expected one ------------------
_EXPECTED = {
 "1.1": "bdf83046b27630fb",
 "1.2": "fb9b8bcd4c02eaf2",
 "1.3": "c75cb66ae28d8ebc",
 "1.4": "5feceb66ffc86f38",
 "1.5": "fbe86ebdfe081051",
 "1.6": "e6da2ed23244cd5d",
 "1.7": "6767a9a3437def51",
 "1.8": "56855b425714a7bc",
 "1.9": "4b227777d4dd1fc6",
 "1.10": "2fafdfa88f249b4c",
 "1.11": "6efcd79928ae33a2",
 "1.12": "4e3970018fa50092"
}
_RECORD = os.environ.get("LAB_RECORD")

def _norm(v):
    if isinstance(v, bool) or v is None: return v
    if isinstance(v, int): return v
    if isinstance(v, float): return round(v, 2)
    if isinstance(v, str): return v.strip()
    if isinstance(v, dict): return {str(k): _norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
    if isinstance(v, (list, tuple, set)):
        items = [_norm(x) for x in v]
        return sorted(items, key=json.dumps) if isinstance(v, set) else items
    if hasattr(v, "item"): return _norm(v.item())          # numpy numbers
    raise TypeError(f"check() needs plain Python values, got {type(v).__name__}")

def check(task, value):
    normal = _norm(value)
    if _RECORD:
        rec = json.load(open(_RECORD)) if os.path.exists(_RECORD) else {}
        rec[task] = normal; json.dump(rec, open(_RECORD, "w"))
    ok = hashlib.sha256(json.dumps(normal).encode()).hexdigest()[:16] == _EXPECTED.get(task)
    print(f"✅ {task} correct" if ok else f"❌ {task} not yet: check your code and the task text again")

def check_true(task, condition, hint=""):
    print(f"✅ {task} correct" if condition else f"❌ {task} not yet. {hint}")


downloading yellow_tripdata_2024-01.parquet
downloading taxi_zone_lookup.csv


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/27 18:35:31 WARN Utils: Your hostname, Usmanovs-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 10.174.192.212 instead (on interface en0)
26/10/27 18:35:31 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/27 18:35:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/27 18:35:33 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/10/27 18:35:33 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


Spark 4.2.0 | UI: http://10.174.192.212:4042


## A. The session and the data

**1.1 Which Spark version, master URL and default parallelism is your session using?** Put the three values in *info*.

In [9]:
info = (spark.version, spark.sparkContext.master, spark.sparkContext.defaultParallelism)
info

('4.2.0', 'local[4]', 4)

In [10]:
check("1.1", info)

✅ 1.1 correct


**1.2 Load the trips file `data/yellow_tripdata_2024-01.parquet` into *trips*. How many trips are in the dataset?** Put the number in *n_trips*.

In [13]:
trips = spark.read.parquet("data/yellow_tripdata_2024-01.parquet")
trips.printSchema()
n_trips = trips.count()
n_trips

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



2964624

In [14]:
check("1.2", n_trips)

✅ 1.2 correct


**1.3 Load `data/taxi_zone_lookup.csv` into *zones* using the schema `LocationID int, Borough string, Zone string, service_zone string`. How many zones are in Manhattan (*Borough*)?** Put the number in *n_manhattan*.

In [ ]:
zones = spark.read.option("header", True).schema(___).csv(___)
n_manhattan = ___
n_manhattan

In [ ]:
check("1.3", n_manhattan)

## B. Lazy evaluation: transformations vs. actions

**1.4 Create *long_trips*: trips with *trip_distance* over 10 miles, with the columns *tpep_pickup_datetime*, *PULocationID*, *DOLocationID*, *trip_distance*, *fare_amount* and a new column *duration_min* (minutes from pickup to dropoff). How many Spark jobs does creating it start?**

Guess first, then measure with `jobs_run`.

In [ ]:
# My prediction: ___ jobs
def build():
    return (trips.filter(___)
                 .select(___)
                 .withColumn("duration_min", ___))

jobs_building = jobs_run(build)
long_trips = build()
jobs_building

In [ ]:
check("1.4", jobs_building)

**1.5 Which of these start a Spark job on *long_trips*: `printSchema`, `explain`, `columns`, `count`, `show`, writing to the `noop` sink?**

Guess first, then measure each with `jobs_run`.

In [ ]:
job_counts = {
    "printSchema": jobs_run(lambda: long_trips.printSchema()),
    "explain": jobs_run(lambda: ___),
    "columns": jobs_run(lambda: ___),
    "count": jobs_run(lambda: ___),
    "show": jobs_run(lambda: long_trips.show(5)),
    "write_noop": jobs_run(lambda: long_trips.write.format("noop").mode("overwrite").save()),
}
job_counts

In [ ]:
check("1.5", job_counts)

**1.6 What is the average *duration_min* of *long_trips*?** Round to 1 decimal and put it in *avg_long*.

In [ ]:
check("1.6", avg_long)

## C. Narrow vs. wide: stages and tasks

**1.7 How many stages does each query run?**
* a: trips with *fare_amount* over 100, only *fare_amount* and *tip_amount*, written to the `noop` sink
* b: total *total_amount* per *payment_type*, collected
* c: all trips sorted by *fare_amount*, written to the `noop` sink

Guess first, then measure with `stages_run`. Also measure how many jobs query c starts (*jobs_c*).

In [ ]:
noop = lambda df: df.write.format("noop").mode("overwrite").save()   # given: write without saving



In [ ]:
check("1.7", [stages, jobs_c])

**1.8 How many tasks does the last stage of query b have with the default `spark.sql.shuffle.partitions`, and with it set to 8?** Use `last_stage_tasks`, then set it back to 200.

In [ ]:
check("1.8", tasks)

**1.9 How many partitions does *trips* have?** Put it in *read_tasks*, then find the same number of tasks in the first stage of query b in the Spark UI.

In [ ]:
check("1.9", read_tasks)

## D. One real question, end to end

**1.10 Which 5 pickup boroughs had the most trips in January 2024?**
* count only pickups from `2024-01-01` up to `2024-02-01` (*tpep_pickup_datetime*)
* join *trips.PULocationID* to *zones.LocationID*
* *top_boroughs* = list of `(Borough, count)`, largest first

Then measure the jobs and stages of collecting it into *jobs_stages*.

In [ ]:
check("1.10", [top_boroughs, jobs_stages])

**1.11 Write the app *top_zones.py* that finds the 3 pickup zones (*Zone*) with the most January 2024 trips and saves them as CSV to `out/top_zones`. Run it with `spark-submit`.**

In [ ]:
%%writefile top_zones.py
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName("top-zones").getOrCreate()
trips = spark.read.parquet("data/yellow_tripdata_2024-01.parquet")
zones = spark.read.option("header", True).csv("data/taxi_zone_lookup.csv")
top = ___  # the 3 zones with most January 2024 pickups, columns Zone and count
top.write.mode("overwrite").option("header", True).csv("out/top_zones")
spark.stop()

In [ ]:
SPARK_SUBMIT = os.path.join(os.path.dirname(sys.executable), "spark-submit")
!{SPARK_SUBMIT} --master "local[2]" top_zones.py 2>/dev/null
top_zones = [r["Zone"] for r in spark.read.option("header", True).csv("out/top_zones").orderBy(F.desc(F.col("count").cast("int"))).collect()]
check("1.11", top_zones)

## E. Fault tolerance

**1.12 How many times does Spark try a task that keeps failing: with master `local[4]`, and with `local[4,3]`?**
* *risky* raises an error for fares over $2,000; run `trips.agg(F.sum(risky("fare_amount"))).collect()` and catch the error
* read the failed attempts with `failed_attempts()`
* restart with `new_session("local[4,3]")`, reload *trips*, run it again

(Why not `trips.select(risky(...)).count()`? `count()` needs no columns, so Spark never calls the UDF.)

In [ ]:
from pyspark.sql.functions import udf
risky = udf(lambda fare: 1 / 0 if fare is not None and fare > 2000 else fare, "double")   # given



In [ ]:
check("1.12", attempts)

---
Done? In the Theory part of Day 1, answer the red checklist cards aloud, pointing at what you measured here.